# Build a Chest X-ray Classifier in 30 Minutes

**Medic Dojo · Free Masterclass**

In the next 30 minutes you will train a real deep-learning model to distinguish normal chest X-rays from ones showing effusion, on real medical imaging data.

You do not need to understand every line. Just click ▶ on each cell in order. Read the comments as you go — that is where the teaching lives. If you want to poke at anything, edit it — the notebook is yours.

**Before you start:**
1. Open the *Runtime* menu → *Change runtime type* → set **Hardware accelerator** to **GPU** (T4 is fine).
2. Click ▶ on the first code cell below to begin.

Data: a balanced 500-image subset of the [NIH ChestX-ray14 dataset](https://www.kaggle.com/datasets/nih-chest-xrays/data), preprocessed for this masterclass.

Runtime: ~10 minutes of training + ~5 minutes of setup. Total ~30 minutes including reading.


## 1. Check the environment

Colab already has PyTorch, torchvision, scikit-learn and matplotlib installed. This cell just confirms you have a GPU. If it prints `cpu` instead of `cuda`, go to *Runtime → Change runtime type* and pick a GPU.

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Download the dataset

We are pulling ~500 chest X-rays that have been resized to 224×224 pixels and split into a `train/` folder (400 images) and a `test/` folder (100 images). Each folder has two sub-folders: `normal/` and `effusion/`. That folder structure is important — it is the convention PyTorch expects for image classification, and it means we can load everything with a single line later on.

The download is ~40 MB and takes about 10 seconds.

In [ ]:
import os
import urllib.request
import zipfile

DATA_URL = 'https://z7zde9q76ypjbfjk.public.blob.vercel-storage.com/masterclass/chest-xray-subset-icLKNQNf6s6GQWabmOUrywk049qN2M.zip'
DATA_DIR = 'chest-xray-subset'
ZIP_PATH = 'chest-xray-subset.zip'

if not os.path.exists(DATA_DIR):
    print('Downloading dataset...')
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    print('Extracting...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('.')
    os.remove(ZIP_PATH)

print('Done. Folder structure:')
for split in ['train', 'test']:
    for cls in ['normal', 'effusion']:
        path = os.path.join(DATA_DIR, split, cls)
        n = len(os.listdir(path)) if os.path.isdir(path) else 0
        print(f'  {split}/{cls}: {n} images')

## 3. Look at a few X-rays

Before we do anything else — actually look at the data. This is the single most under-taught habit in applied ML. If you cannot tell the difference between the two classes with your own eyes, no model will save you.

The row below is 8 *normal* chest X-rays; the row after is 8 films showing *pleural effusion*. Look for blunting of the costophrenic angles and dense homogeneous opacities at the lung bases in the effusion row.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os, random

def show_row(folder, title, n=8):
    files = os.listdir(folder)
    random.seed(42)
    sample = random.sample(files, min(n, len(files)))
    fig, axes = plt.subplots(1, n, figsize=(20, 3))
    for ax, f in zip(axes, sample):
        img = Image.open(os.path.join(folder, f)).convert('L')
        ax.imshow(img, cmap='gray')
        ax.axis('off')
    fig.suptitle(title, fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

show_row(f'{DATA_DIR}/train/normal', 'NORMAL')
show_row(f'{DATA_DIR}/train/effusion', 'EFFUSION')

## 4. Prepare the data pipeline

A *data pipeline* is what feeds images to the model during training. Three things happen here:

1. **Transforms** — we standardise every image the same way. Resize to 224×224, convert to a tensor of pixel values in [0, 1], then normalise using the mean and standard deviation of the ImageNet dataset that our pre-trained model was originally trained on. This is important: if we don't normalise the way the model expects, it sees garbage.
2. **Dataset** — `ImageFolder` reads our folder structure and assigns class 0 to `normal/` (comes first alphabetically) and class 1 to `effusion/`.
3. **DataLoader** — wraps the dataset in an iterator that hands the model a *batch* of 16 images at a time. Batching is faster than one-at-a-time; it also stabilises training.

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ImageNet statistics — required because our pre-trained model expects inputs
# normalised in the same way it was originally trained.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),  # ResNet expects 3 channels; X-rays are grayscale
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = datasets.ImageFolder(f'{DATA_DIR}/train', transform=transform)
test_ds  = datasets.ImageFolder(f'{DATA_DIR}/test',  transform=transform)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=16, shuffle=False, num_workers=2)

print(f'Classes: {train_ds.classes}')  # ['normal', 'effusion']
print(f'Training images: {len(train_ds)}')
print(f'Test images:     {len(test_ds)}')

## 5. Load a pre-trained ResNet-18 and adapt it

**ResNet-18** is a 18-layer convolutional neural network from 2015 that changed computer vision. It has already been trained on 1.28 million natural images (dogs, cars, birds — the ImageNet dataset). This is important: it already knows how to see edges, textures, shapes.

We are using **transfer learning** — we take that pre-trained model and swap its very last layer for our own 2-class output (normal vs effusion). Then we fine-tune the whole network on our small chest X-ray dataset. This is orders of magnitude more efficient than training from scratch, and it works well even with only a few hundred images.

This is exactly the pattern most published clinical AI papers use.

In [ ]:
import torch.nn as nn
from torchvision import models

# Load the pre-trained network. On first run, torchvision downloads the weights (~45 MB).
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Replace the final layer: originally outputs 1000 ImageNet classes; we want 2.
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)

model = model.to(device)
print(f'Model on {device}. Final layer:')
print(model.fc)

## 6. Train the model

*Training* means: repeatedly show the model batches of images, compare its predictions to the correct labels, and nudge every internal weight in the direction that would have produced a better answer. Do this thousands of times and the model gradually learns.

We will do **3 epochs** (one epoch = one full pass through the training set). On a Colab T4 GPU this takes ~2 minutes per epoch. Watch the loss go down and the accuracy go up.

- **Loss function**: cross-entropy — standard for classification. Loss going down = model getting better.
- **Optimiser**: Adam — the standard modern optimiser. Handles the learning-rate scheduling for us.
- **Learning rate**: `1e-4` — small enough not to disturb the pre-trained features too much.

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

EPOCHS = 3
for epoch in range(EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        _, preds = outputs.max(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    print(f'Epoch {epoch+1}/{EPOCHS} — loss: {running_loss/total:.4f} — train accuracy: {correct/total:.2%}')

print('\nTraining complete.')

## 7. Evaluate on the held-out test set

The model has never seen these 100 test images during training. This is the only honest measure of whether it has actually learned anything useful — or just memorised the training set.

We compute four things:
- **Accuracy** — overall % correct. Easy to interpret but misleading when classes are imbalanced.
- **Sensitivity (recall for effusion)** — of all real effusion cases, what fraction did we catch? A missed effusion is a false negative.
- **Specificity** — of all real normals, what fraction did we correctly leave alone? A false positive is a normal patient flagged for further tests.
- **AUC (Area Under the ROC Curve)** — a single-number summary of how well the model separates the two classes across all possible decision thresholds. 1.0 is perfect, 0.5 is coin-flip. Anything above 0.85 is a decent clinical model in the literature.

In [ ]:
import numpy as np
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix

model.eval()
all_labels, all_scores, all_preds = [], [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        probs = torch.softmax(outputs, dim=1)[:, 1]  # probability of 'effusion'
        _, preds = outputs.max(1)
        all_labels.extend(labels.numpy())
        all_scores.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

all_labels = np.array(all_labels)
all_scores = np.array(all_scores)
all_preds  = np.array(all_preds)

tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()
accuracy    = (tp + tn) / (tp + tn + fp + fn)
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
auc         = roc_auc_score(all_labels, all_scores)

print(f'Accuracy:    {accuracy:.2%}')
print(f'Sensitivity: {sensitivity:.2%}   (of real effusions, this fraction caught)')
print(f'Specificity: {specificity:.2%}   (of real normals, this fraction correctly left alone)')
print(f'AUC:         {auc:.3f}')

## 8. Plot the ROC curve

The **ROC curve** shows every possible trade-off between sensitivity and 1-specificity that this model can produce, depending on where you set the decision threshold. A model that is perfect hugs the top-left corner. A model that is useless sits on the diagonal.

The **AUC** is literally the area under this curve — a single number summary.

In [ ]:
fpr, tpr, _ = roc_curve(all_labels, all_scores)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, color='#10b981', linewidth=2.5, label=f'Your model (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], color='#666', linestyle='--', linewidth=1, label='Random guessing')
plt.xlabel('False positive rate (1 − specificity)')
plt.ylabel('True positive rate (sensitivity)')
plt.title('ROC curve — your chest X-ray classifier')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved as roc_curve.png — download from the file browser on the left.')

## 9. Make a prediction on a novel X-ray

One more thing. Let's pick a random image the model has never seen and watch it make a prediction.

In [ ]:
import random

idx = random.randint(0, len(test_ds) - 1)
img_tensor, true_label = test_ds[idx]

model.eval()
with torch.no_grad():
    output = model(img_tensor.unsqueeze(0).to(device))
    probs = torch.softmax(output, dim=1).squeeze().cpu().numpy()

pred_class  = int(probs.argmax())
confidence  = float(probs[pred_class])
class_names = train_ds.classes  # ['normal', 'effusion']

img_path = test_ds.samples[idx][0]
plt.figure(figsize=(6, 6))
plt.imshow(Image.open(img_path).convert('L'), cmap='gray')
plt.axis('off')
correct = '✓' if pred_class == true_label else '✗'
plt.title(
    f'True: {class_names[true_label]}   |   Predicted: {class_names[pred_class]} ({confidence:.1%})   {correct}',
    fontsize=12,
)
plt.tight_layout()
plt.show()

## 10. Get your certificate

You did it. You trained a real chest X-ray classifier. Type your name in the cell below to generate a certificate with your final AUC — you can download it and post it on LinkedIn.

Tag [@medicdojo](https://linkedin.com/company/medicdojo) if you do — we love seeing this.

In [ ]:
YOUR_NAME = 'Dr. Your Name'  # ← replace with your name, then run this cell

from PIL import Image, ImageDraw, ImageFont
from datetime import date

W, H = 1600, 1000
cert = Image.new('RGB', (W, H), '#000000')
draw = ImageDraw.Draw(cert)

def load_font(size, bold=False):
    for path in [
        '/usr/share/fonts/truetype/dejavu/DejaVuSerif-Bold.ttf' if bold else '/usr/share/fonts/truetype/dejavu/DejaVuSerif.ttf',
        '/usr/share/fonts/truetype/liberation/LiberationSerif-Bold.ttf' if bold else '/usr/share/fonts/truetype/liberation/LiberationSerif-Regular.ttf',
    ]:
        try: return ImageFont.truetype(path, size)
        except OSError: continue
    return ImageFont.load_default()

def draw_centred(text, y, size, color='#F2EFE9', bold=False, tracking=0):
    font = load_font(size, bold)
    if tracking:
        spaced = (' ' * tracking).join(list(text))
        bbox = draw.textbbox((0, 0), spaced, font=font)
        draw.text(((W - (bbox[2] - bbox[0])) / 2, y), spaced, fill=color, font=font)
    else:
        bbox = draw.textbbox((0, 0), text, font=font)
        draw.text(((W - (bbox[2] - bbox[0])) / 2, y), text, fill=color, font=font)

draw_centred('MEDIC  DOJO', 100, 22, '#10b981', bold=True, tracking=1)
draw_centred('CHEST X-RAY CLASSIFIER MASTERCLASS', 145, 18, '#B8B8B8', tracking=1)
draw.line([(600, 260), (1000, 260)], fill='#10b981', width=2)
draw_centred('Awarded to', 310, 28, '#B8B8B8')
draw_centred(YOUR_NAME, 370, 64, '#F2EFE9', bold=True)
draw_centred('for training a working chest X-ray classifier', 500, 28, '#D8D8D8')
draw_centred('on real medical imaging data', 540, 28, '#D8D8D8')
draw_centred(f'Final test-set AUC: {auc:.3f}', 660, 46, '#10b981', bold=True)
draw_centred(date.today().strftime('%d %B %Y'), 780, 22, '#888', tracking=1)
draw_centred('medicdojo.com/masterclass', 830, 20, '#888')

cert.save('my_certificate.png')
print('Saved as my_certificate.png — download from the file browser on the left.')
cert

## What just happened — and what to do next

In the last 30 minutes you:

- Loaded a pre-trained ResNet-18 (millions of parameters).
- Fine-tuned it on 400 real chest X-rays using transfer learning.
- Evaluated it properly — sensitivity, specificity, AUC, ROC.
- Watched it predict on unseen data.

This is the same pattern used by the majority of published clinical AI papers. You now understand more about how these models work than most clinicians using them in practice.

---

### Want the line-by-line walkthrough?

If any cell surprised you, or you'd like a plain-English explanation of exactly what every line of code did and what it means for real clinical validation — we've written a detailed 4-page PDF called **The Explainer** that walks through every cell in this notebook.

👉 **[Get The Explainer, free →](https://medicdojo.com/masterclass/mark-scheme)**

Just leave your email and it'll land in your inbox in a minute.

---

### What this masterclass did *not* cover (but the full course does)

- How to build a model from scratch rather than fine-tune a pre-trained one.
- How to detect and handle the shortcuts, biases and failure modes real clinical data introduces.
- How to validate a model in a way that would actually be defensible in a clinical setting — external validation, subgroup analysis, calibration.
- How to critically appraise any AI paper or commercial tool using a rigorous framework.
- How to build a working clinical predictor on tabular data (not just images) — the format most audits and QI projects use.

The founding cohort opens **October 2026**. Enrol now for **20% off** — offer closes 8 September 2026.

👉 [See the full course](https://medicdojo.com/ml)

---

*Questions on any specific cell? Reply to the confirmation email that The Explainer arrives in — it comes straight to Samar.*